# LingBot 6k → Colab A100 Stage-2
状态：迁移准备版，CUDA硬件检查已在现有Colab通过，LingBot训练/仿真尚未通过。
保留原6k结构、归一化与动作配置；LoRA新优化器从第0步开始。不要执行Run all，逐节确认。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE=Path('/content/drive/MyDrive/LingBot-Colab-Stage2-20261008')
LOCAL=Path('/content/lingbot-stage2')
LOCAL.mkdir(parents=True,exist_ok=True)
import subprocess, json, shutil, os
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True))
print('Local disk free GiB:',shutil.disk_usage('/content').free/2**30)


## 输入校验与本地缓存
Drive必须包含full_sft_6000完整15文件、migration_package.tar.gz和lerobot完整数据目录。只复制已完整校验的文件；不使用本地未完成44%分块。先确认输入齐全，再安装环境。

In [ ]:
import tarfile
archive=DRIVE/'migration_package.tar.gz'
assert archive.is_file(), '迁移源码包尚未上传'
with tarfile.open(archive) as f: f.extractall(LOCAL,filter='data')
source=DRIVE/'full_sft_6000'
index=json.loads((source/'model.safetensors.index.json').read_text())
assert all((source/x).is_file() for x in set(index['weight_map'].values())), '权重未上传完'
assert (DRIVE/'lerobot/put_object_cabinet_joint_v30/meta/info.json').is_file(), '缺完整cabinet LeRobot数据；先补齐或按原转换配方重建'
shutil.copytree(source,LOCAL/'full_sft_6000',dirs_exist_ok=True)
shutil.copytree(DRIVE/'lerobot',LOCAL/'lerobot',dirs_exist_ok=True)


## CUDA环境
官方Python3.12/PyTorch2.8.0/FlashAttention2.8.3，独立环境；不安装ROCm依赖。此安装流程尚未在本Colab验证。

In [ ]:
subprocess.run(['bash',str(LOCAL/'package/setup_cuda.sh')],check=True)

## Teacher下载
首次smoke保留原结构与辅助损失，故需要teacher；不清空align_params。后续若移除辅助目标，须单独做结构保持补丁和权重加载对照。

In [ ]:
PY='/content/lingbot-venv/bin/python'
subprocess.run([PY,'-c',"from huggingface_hub import snapshot_download; snapshot_download('robbyant/lingbot-vla-v2-6b', revision='11c703bf6a5c1f45b3b69168482da11fdbba53d7',allow_patterns=['depth/model.pt','dino_video/teacher_step_10000.pth','dino_video/config.yaml'],local_dir='/content/lingbot-stage2/teachers/base'); snapshot_download('Ruicheng/moge-2-vitb-normal',revision='ca5f0e07ff01d3e5a364c1d954ed12ee1814b368',allow_patterns=['model.pt'],local_dir='/content/lingbot-stage2/teachers/moge')"],check=True)


## 两步LoRA训练smoke
要求真实数据、有限loss/梯度、非零LoRA训练参数，以及global_step_2断点。日志直接保存在Drive。原DCP保存器也保存冻结参数，因此断点可能超过20GiB，不能视作小型adapter。保存耗时须计入预算。

In [ ]:
subprocess.run([PY,str(LOCAL/'package/prepare.py'),'--drive',str(DRIVE),'--mode','smoke'],check=True)
env=os.environ.copy()
for k in ['HIP_VISIBLE_DEVICES','ROCR_VISIBLE_DEVICES','AITER_TRITON_ONLY','FLASH_ATTENTION_TRITON_AMD_ENABLE']: env.pop(k,None)
env.update(CUDA_VISIBLE_DEVICES='0',PYTHONUNBUFFERED='1',QWEN3VL_PATH=str(LOCAL/'package/qwen'),PYTHONPATH=str(LOCAL/'package/lingbot/lingbotvla/models/vla/vision_models/morgbd_clean/3rd/utils3d'))
cmd=[PY,'-m','torch.distributed.run','--standalone','--nproc-per-node=1','-m','tasks.vla.train_lingbotvla',str(LOCAL/'smoke.yaml')]
log=DRIVE/'runs/smoke.log'
with log.open('w') as f:
 p=subprocess.Popen(cmd,cwd=LOCAL/'package/lingbot',env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
 for line in p.stdout: print(line,end=''); f.write(line); f.flush()
 assert p.wait()==0, f'训练失败；检查{log}'
assert (DRIVE/'runs/smoke/global_step_2/model/.metadata').is_file(), '缺model DCP'
assert (DRIVE/'runs/smoke/global_step_2/optimizer/.metadata').is_file(), '缺optimizer DCP'
print('请核对loss与LoRA参数日志，再进行同一单卡Stage2断点恢复测试；目前不自动开始长训练。')


## Stage-2与评测门槛
先单卡保存/恢复到step3验证，再启动500步试验；初始micro batch=1、累积4、rank8、action attention LoRA、lr2e-5。目标任务可在prepare.py --target TASK --repeat 3重复，实为按帧加权，必须报告实际混合比例。
原50任务训练总量必须核对2500 episodes/549787 frames；原始与增强来源保持同组。新增强尚未物理验收，默认不混入。
闭环需另外安装CUDA兼容SAPIEN/MPLib并补齐assets，保持冻结seed、seen指令、valid-episode retry与官方成功判据。建议cabinet 2回合→3难任务+2锚任务各10回合→剩余预算决定全50任务；CUDA仿真差异必须重新跑6k对照。当前未运行评测，不能报告提升。
## 恢复规则
8 GPU DCP模型/优化器张量理论可重新分片，但rank-specific extra_state/dataloader需单独处理；不直接传原71GiB DCP给单卡Stage2。现有HF无需转换。新单卡Stage2恢复使用原配方、同一输出副本和明确--train.load_checkpoint_path；检查global step、非空AdamW状态以及没有Starting training from scratch/Skip loading Optimizer。
## 30小时预算
先测速20步并记录峰值显存及保存耗时；约2–4小时环境/数据、1小时smoke/恢复、16–20小时训练、5–7小时评测与保存只是候选，按实测缩减。Colab运行时与GPU可用时长不作保证。